In [1]:
!pip install -q fastai kaggle scikit-learn seaborn matplotlib pandas numpy

In [3]:
from fastai.vision.all import *
from fastai.callback.fp16 import *

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn  as sns

from sklearn.metrics import confusion_matrix,classification_report,f1_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from pathlib import Path
import os
import zipfile
import shutil

In [4]:
BASE_PATH = Path("data")
DOGS_CATS_PATH = BASE_PATH / "dogs-vs-cats"
DOG_BREED_PATH = BASE_PATH / "dog-breed-identification"

OUTPUT_PATH = Path("outputs")
MODEL_PATH = OUTPUT_PATH / "models"
SUBMISSION_PATH = OUTPUT_PATH / "submissions"
FIGURE_PATH = OUTPUT_PATH / "figures"

for p in [BASE_PATH, DOGS_CATS_PATH, DOG_BREED_PATH, OUTPUT_PATH, MODEL_PATH, SUBMISSION_PATH, FIGURE_PATH]:
    p.mkdir(parents=True, exist_ok=True)

In [5]:
!kaggle competitions download -c dogs-vs-cats-redux-kernels-edition -p data/dogs-vs-cats
!kaggle competitions download -c dog-breed-identification -p data/dog-breed-identification

Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token
Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, gener

In [6]:
def unzip_all_files(folder):
    folder = Path(folder)
    for zip_file in folder.glob("*.zip"):
        print("Unzipping:", zip_file)
        with zipfile.ZipFile(zip_file, "r") as z:
            z.extractall(folder)

unzip_all_files(DOGS_CATS_PATH)
unzip_all_files(DOG_BREED_PATH)

In [7]:
!kaggle competitions download

Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token


In [ ]:
cats_dogs_train = DOGS_CATS_PATH/ 'train'


if not cats_dogs_train.exists():
    train_zip = DOGS_CATS_PATH/'train.zip'
    with zipfile.ZipFile(train_zip,'r') as z:
        z.extractall(DOGS_CATS_PATH)
    
files = get_image_files(cats_dogs_train)

len(files), files[:5]

In [ ]:
def get_cat_dog_label(path):
    name = path.name
    return name.split('.')[0]

In [ ]:
cat_dog_dls = ImageDataLoaders.form_name_func(
    path = cats_dogs_train,
    fname= files,
    label_func = get_cat_dog_label,
    valid_pca= 0.2,
    seed=6,
    item_tfms = Resize(224),
    batch_tfms = aug_transforms(
        max_rotate=10,
        max_zoom=1.1,
        max_lighting=0.2,
        max_warp=0.1
    ),
    bs = 32
)

In [ ]:
cat_dog_dls.show_batch(max_n=9,figsize=(7,7))

In [ ]:
cat_dog_learner = vision_learner(
    cat_dog_dls,
    resnet34,
    metrics=[accuracy,F1Score()]
)

In [ ]:
cat_dog_learner.lr_find()

In [ ]:
cat_dog_learner.fine_tune(3,base_lr =1e-3)

In [ ]:
cat_dog_learner.export(MODEL_PATH/'cat_dog_classifier.pkl')

In [ ]:
interp = ClassificationInterpretation.from_learner(cat_dog_learner)

interp.plot_confusion_matrix(figsize=(6,6))


In [ ]:
interp.plot_top_losses(9,figsize=(10,10))

In [ ]:
labels_df = pd.read_csv(DOG_BREED_PATH/'labels.csv')

labels_df.head()

In [ ]:
labels_df['breed'].nunique()

In [ ]:
plt.figure(figsize=(12,5))
labels_df['bread'].value_counts().head(30).plot(kind='bar')
plt.title('Top 30 Dog  Breeds By Image Count')
plt.show()


In [ ]:
labels_df['filename'] = labels_df['id'].apply(lambda x: f'{x}.jpg')
labels_df.head()